In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/rhome/sawale/indus_traning/mlm-fine-tuning/mlm')

In [3]:
from utils import generate_inference, printd
import json
from preprocess_data import preprocess_dataset

/rhome/sawale/indus_traning/mlm-fine-tuning/xenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
config_path = "/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/config.json"
data_src = "local"
n_rows = None

with open(config_path, "r") as file:
    config = json.load(file)

lm_dataset, tokenizer, data_collator = preprocess_dataset(
        config.get("input"),
        data_src,
        n_rows,
        chunksize=1024,
    )

lm_dataset

DatasetDict({
    train: Dataset({
        features: ['url_1', 'first_field', 'second_field', 'third_field', 'fourth_field', 'text', 'normalized_url', 'collection__config_folder', 'url_2', 'generated_title', 'scraped_title', 'division_display', 'document_type_display'],
        num_rows: 96992
    })
    validation: Dataset({
        features: ['url_1', 'first_field', 'second_field', 'third_field', 'fourth_field', 'text', 'normalized_url', 'collection__config_folder', 'url_2', 'generated_title', 'scraped_title', 'division_display', 'document_type_display'],
        num_rows: 12124
    })
    test: Dataset({
        features: ['url_1', 'first_field', 'second_field', 'third_field', 'fourth_field', 'text', 'normalized_url', 'collection__config_folder', 'url_2', 'generated_title', 'scraped_title', 'division_display', 'document_type_display'],
        num_rows: 12124
    })
})


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 153019
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 18371
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 20116
    })
})

In [ ]:
# using 512 context window
# 96992 + 12124 + 12124 = 121240
# 242482 + 28821 + 32173 = 303476

# using 1024 context window
# 153019 + 18371 + 20116 = 191506

In [42]:
inference_df = generate_inference(
        lm_dataset["test"],
        tokenizer,
        model_save_loc="/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/model_outputs/20250207_23-03-37/model",
        top_k=config.get("output").get("inference").get("top_k"),
        n_predictions=config.get("output").get("inference").get("n_predictions"),
        max_length=config.get("input").get("dataset").get("chunk_size"),
    )

Device set to use cuda:0


In [43]:
inference_df

,input,target,top1,top2,top3
0,[CLS]\nSkip to main content\n\nGoddard Earth S...,for,"{'score': 0.9288, 'token_str': ' for', 'token'...","{'score': 0.036, 'token_str': ' after', 'token...","{'score': 0.012, 'token_str': ' following', 't..."
1,in:\n\nI am interested in: Undergraduate\n\nI...,Professional,"{'score': 0.9998, 'token_str': ' Professional'...","{'score': 0.0001, 'token_str': ' Graduate', 't...","{'score': 0.0, 'token_str': ' Major', 'token':..."
2,[CLS]\nSkip to main content\n\nGoddard Earth S...,ission,"{'score': 0.9728, 'token_str': 'ission', 'toke...","{'score': 0.0129, 'token_str': 'otive', 'token...","{'score': 0.0092, 'token_str': 'atter', 'token..."
3,\n\nImportant Contacts\n\nOnline Directory\n\n...,\n,"{'score': 0.5406, 'token_str': '/', 'token': 16}","{'score': 0.4416, 'token_str': ' ', 'token': 187}","{'score': 0.0065, 'token_str': ' Show', 'token..."
4,[CLS]\nSkip to main content\n\nGoddard Earth S...,�,"{'score': 0.1601, 'token_str': '�', 'token': 120}","{'score': 0.1439, 'token_str': '�', 'token': 227}","{'score': 0.0812, 'token_str': '<<', 'token': ..."
5,[CLS]\nSkip to main content\n\nGoddard Earth S...,28,"{'score': 0.9415, 'token_str': '28', 'token': ...","{'score': 0.038, 'token_str': '50', 'token': 1...","{'score': 0.0032, 'token_str': '30', 'token': ..."
6,Text Alerts\n\nContact Us\n\nRequest Info\n\n...,�,"{'score': 0.9999, 'token_str': ' �', 'token': ...","{'score': 0.0001, 'token_str': '�', 'token': 325}","{'score': 0.0, 'token_str': ' ', 'token': 187}"
7,[CLS]\nTOOLKIT\n\nCONTESTS\n\nWEBINARS\n\nGET ...,ED,"{'score': 0.9999, 'token_str': 'ED', 'token': ...","{'score': 0.0, 'token_str': 'OUS', 'token': 27...","{'score': 0.0, 'token_str': 'E', 'token': 38}"
8,[CLS]\nSkip to main content\n\nMenu\n\nHome\n\...,.,"{'score': 1.0, 'token_str': '.', 'token': 15}","{'score': 0.0, 'token_str': '.', 'token': 964}","{'score': 0.0, 'token_str': '\.', 'token': 15461}"
9,and Observation to Benefit the Environment (G...,",","{'score': 0.9884, 'token_str': ',', 'token': 13}","{'score': 0.0026, 'token_str': ' night', 'toke...","{'score': 0.0023, 'token_str': ' evening', 'to..."


In [27]:
from utils import mask_random_token
from transformers import PreTrainedModel, PreTrainedTokenizer, pipeline
model_save_loc = "/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/model_outputs/20250207_23-03-37/model"

n_predictions = 2000
datasplit = lm_dataset["test"]

if n_predictions is not None:
    datasplit = datasplit.select(range(min(datasplit.num_rows, n_predictions)))

masked_dataset = datasplit.map(
    lambda example: mask_random_token(example, tokenizer),
)
decoded_texts = [
    tokenizer.decode(example["input_ids"], clean_up_tokenization_spaces=True)
    for example in masked_dataset
]

mask_filler = pipeline("fill-mask", model=model_save_loc, tokenizer=tokenizer)

# filter decoded_texts: remove those without <mask> token
decoded_texts = [text for text in decoded_texts if tokenizer.mask_token in text]
masked_token_strs = [
    example.get("masked_token_str")
    for example, text in zip(masked_dataset, decoded_texts)
    if tokenizer.mask_token in text
]

Map: 100%|██████████| 509/509 [00:04<00:00, 121.51 examples/s]
Device set to use cuda:0


In [35]:
decoded_texts[5]

'[CLS]\nSkip to main content\n\nGoddard Earth Sciences Technology and Research (GESTAR) II\n\nMenu\n\nSearch\n\nSearch Context This Site All of UMBC\n\nGoddard Earth Sciences Technology and Research (GESTAR) II\n\nHome\n\nAbout GESTAR II\n\nMission and Vision Statements\n\nLeadership\n\nPartners\n\nGESTAR II Directory\n\nResearchers A-F\n\nResearchers G-K\n\nResearchers L-Q\n\nResearchers R-Z\n\nGraduate Research Assistants\n\nAdministrative Personnel\n\nGESTAR II Organization Chart\n\nNews & Reports\n\nNews & Highlights\n\nGESTAR II Annual Reports\n\nEvents\n\nSeminars & Colloquia\n\nAERONET Science and Application Exchange\n\nGESTAR II Seminar Series\n\nGESTAR-II Graduate Student Colloquium\n\nJobs at GESTAR II\n\nLinks for GESTAR II Faculty\n\nGESTAR II Student Opportunities\n\nGESTAR II Graduate Fellowship 2022-2024\n\nGESTAR II Graduate Fellows 2023-2024\n\nGESTAR II MSU Undergraduate Fellowship\n\nGESTAR II MSU Fellows 2023-2024\n\nGESTAR II UMBC Graduate Fellowship 2024-2025\n\n

In [34]:
top_k = 5
max_length = 512


for i, m in enumerate(decoded_texts):
    try:
        results = mask_filler(
                m,
                top_k=top_k,
                tokenizer_kwargs={
                    "truncation": True,
                    "max_length": max_length,
                    "add_special_tokens": False,
                },
            )
    except:
        print(i)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


5
9
15
18
21
33
37
39
49
50
52
53
59
61
63
66
69
71
74
76
78
82
87
89
93
94
96
98
99
100
101
102
103
106
107
109
110
111
113
114
116
118
119
120
121
122
127
128
130
131
132
135
137
139
140
141
142
143
144
145
146
147
148
149
153
158
159
160
163
165
167
170
173
175
178
179
180
181
182
185
186
189
190
193
195
199
201
202
203
204
205
206
211
219
220
221
222
225
226
227
230
232
236
241
242
243
245
246
247
248
249
250
253
254
255
260
263
264
265
267
268
270
271
272
273
274
277
278
279
280
281
283
285
286
287
289
291
292
295
296
297
300
302
303
304
309
311
315
318
319
320
321
322
323
325
327
331
332
333
334
336
337
338
340
341
342
343
344
345
346
349
351
353
354
355
356
358
361
363
365
368
371
372
373
374
376
377
378
379
383
386
389
391
394
396
397
398
399
402
407
409
412
413
414
415
417
418
421
422
423
424
425
426
427
429
430
433
434
435
436
438
439
442
443
445
448
452
453
455
458
459
466
468
473
482
484
486
494
499
501
504


In [ ]:
import torch
from transformers import pipeline
from pprint import pprint

pipe = pipeline(
    "fill-mask",
    model=model,
    torch_dtype=torch.bfloat16,
)

input_text = """[CLS]\nSkip to main content\n\nGoddard Earth Sciences Technology and Research (GESTAR) II\n\nMenu\n\nSearch\n\nSearch Context This Site All of UMBC\n\nGoddard Earth Sciences Technology and Research (GESTAR) II\n\nHome\n\nAbout GESTAR II\n\nMission and Vision Statements\n\nLeadership\n\nPartners\n\nGESTAR II Directory\n\nResearchers A-F\n\nResearchers G-K\n\nResearchers L-Q\n\nResearchers R-Z\n\nGraduate Research Assistants\n\nAdministrative Personnel\n\nGESTAR II Organization Chart\n\nNews & Reports\n\nNews & Highlights\n\nGESTAR II Annual Reports\n\nEvents\n\nSeminars & Colloquia\n\nAERONET Science and Application Exchange\n\nGESTAR II Seminar Series\n\nGESTAR-II Graduate Student Colloquium\n\nJobs at GESTAR II\n\nLinks for GESTAR II Faculty\n\nGESTAR II Student Opportunities\n\nGESTAR II Graduate Fellowship 2022-2024\n\nGESTAR II Graduate Fellows 2023-2024\n\nGESTAR II MSU Undergraduate Fellowship\n\nGESTAR II MSU Fellows 2023-2024\n\nGESTAR II UMBC Graduate Fellowship 2024-2025\n\nGESTAR II UMBC Graduate Fellow 2024-2025\n\nGESTAR II Short-Term Research Grants for Graduate Students (ASU, CSU, PSU)\n\nGESTAR II Visiting Fellows 2024\n\nNASA GSFC Information\n\nI3RC 3D Simulator\n\nUNL - VRTM Remote Sensing Testbed\n\nIn this section\n\nNews & Reports\n\nNews & Highlights\n\nGESTAR II Annual Reports\n\nLocation\n\n5523 Research Park Drive Suite #140 Baltimore, MD 21228\n\nContact\n\nPhone: 410-455-8812\n\nFax: 410-455-8806\n\nContact Us\n\n← Back to News List\n\nGESTAR II Researchers receive 2023 HBG Annual Peer Awards\n\nOn Thursday, October 26, 2023, the 2023 Hydrosphere, Biosphere, and Geophysics (HBG) Annual Peer Awards ceremony was held at NASA Goddard Space Flight Center. The Earth Sciences Division\'s HBG consists of Codes 616, 616, 617, 618, and 61A. Congratulations to the following GESTAR II researchers who were award recipients!\n\nScientific Achievement:\n\nGoutam Konapala (617/UMBC): "For innovative applications of machine learning for hydrology."\n\nFadji Maina (617/UMBC): "For outstanding research contributions about the High Mountain Asia region."\n\nElijah Orland (617/UMBC): "For scientific advances in the assessment of near-real time measures of fire pseudoseverity."\n\nAndrew Sayer (616/UMBC): "For being the atmosphere to PACE ocean."\n\nThomas Stanley (617/UMBC): "For outstanding work developing the LHASA 2.0 model."\n\nScientific/Technical Support:\n\nIvona Cetinic\' (616/MSU): "For extraordinary support of PACE science, outreach, and peer mentorship."\n\nBest Publication - First Author Non-Civil Servant:\n\nAnthony Campbell (618/UMBC): "Global hotspots of salt marsh change and carbon emissions."\n\nCampbell, A.D., Fatoyinbo, L., Goldberg, L. and Lagomasino, D., 2022. Global hotspots of salt marsh change and carbon emissions. Nature, 612 (7941), pp.701-706, https://doi.org/10.1038/s41586-022-05355-z.\n\nOutreach:\n\nIan Carroll (616/UMBC): "For bringing new people to science and new science to people."\n\nTags:\n\ngestar2\n\nPosted: October 30, 2023, 2:05 PM\n\nRead Original Post in myUMBC\n\nLocation\n\n5523 Research Park Drive Suite[MASK]140 Baltimore, MD 21228\n\nContact\n\nPhone: 410-455-8812\n\nFax: 410-455-8806\n\nContact Us\n\nUMBC\n\nUniversity of Maryland, Baltimore County 1000 Hilltop Circle, Baltimore, MD 21250\n\nDirections & Parking Information\n\nResources\n\nAlumni\n\nCareer Center\n\nEvents\n\nGet Help\n\nNews\n\nVisit Campus\n\nWork at UMBC\n\nImportant Contacts\n\nOnline Directory\n\nContact UMBC\n\nRequest Support\n\nEmergency Info\n\nUMBC Police : 410-455-5555\n\nSign Up for"""
results = pipe(input_text)
pprint(results)


Device set to use cuda:0


PipelineException: No mask_token ([MASK]) found on the input